# Lab 1 金融爬蟲：把公開資料變成表格

**今天的目標：** 寫程式自動去網路上把**公開的金融資料**抓回來、整理成表格。你會親手產出兩個檔案：
- **`prices.csv`**（證交所股價）→ 之後 pandas / KNN / 決策樹要用的「數據路」原料
- **`news_raw.csv`**（財經新聞 date/title/text）→ 之後情緒打分要用的「文字路」原料

> 📜 **法遵三句：** ① 只抓公開、不用登入的資料；② 只做課堂教學分析、不轉貼全文、不商用；③ 守禮貌頻率（每抓一頁睡 1 秒、只抓前幾則）。robots.txt「告示牌」怎麼看，老師會講。

In [ ]:
# 📦 先跑這一格：指定版本，避免學校電腦裝到不相容的舊版（裝不起來看 README）
!pip install -q requests beautifulsoup4==4.12.3 pandas==2.2.3

In [ ]:
# 老師的小設定：關掉一個無關緊要的套件提醒，讓等一下的輸出乾淨一點（直接跑、不用改）
import warnings
warnings.filterwarnings("ignore")

## 🔧 第 0 步：環境檢查

**預期輸出：** 印出 `requests` 和 `bs4` 的版本號——有版本號就代表套件裝好了。

In [ ]:
import requests, bs4
print("requests", requests.__version__)
print("bs4     ", bs4.__version__)

---
## A・第一抓：網頁就是一坨文字

`requests.get(網址)` ＝ 寫程式「去跟這個網址要資料」。先把中央社財經新聞的 **RSS**（網站給程式讀的「最新文章清單」）抓回來，看兩件事：① 回來的是一坨文字；② 狀態碼 `200` ＝成功。

> 💬 這種公開清單**不用表明身分也抓得到**——先體會最單純的一行抓網頁（到 C2 抓內頁時，我們才會遇到「要表明身分」的情況）。

**預期輸出：** 狀態碼 `200`，接著一段 `<?xml ...>` 開頭的文字。⚠️ 你抓到的內容**跟這裡不一樣是正常的**——新聞每分鐘在變。

In [ ]:
import requests

# RSS 這種公開清單，不帶任何身分也抓得到——最單純的「一行抓網頁」
r = requests.____("https://feeds.feedburner.com/rsscna/finance")   # TODO：用 requests 的哪個方法「去跟網址要資料」？（HTTP 的 GET）
print("狀態碼：", r.status_code)          # 💡 200＝成功、404＝沒這頁、403/429＝被擋
print("--- 回來的內容前 500 個字 ---")
print(r.text[:500])

### A・順便認識一次「出錯長相」

故意去要一個不存在的網址，看 `404` 長什麼樣。常見狀態碼：**200** 成功、**404** 沒這頁、**403 / 429** 被擋（老師會多講幾個）。

**預期輸出：** `404`。

In [ ]:
# 把剛剛那個 RSS 網址故意打錯（feedburner 上不存在的路徑）
bad = requests.get("https://feeds.feedburner.com/rsscna/nonexistent_xyz")
print("故意打錯網址的狀態碼：", bad.____)   # TODO：從回應物件拿「狀態代號」的那個屬性（200/404 那個）

### 📝 小作業 A

1. 把 `r.text[:500]` 改成 `r.text[:1500]`，多看一點——**數數看你看到了幾個 `<item>`？**（每個 `<item>` 就是一則文章）

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
print(r.text[:1500])
# 整份 RSS 就是一串 <item> 排下來，每個 <item> 裡有 <title>（標題）和 <link>（內頁網址）。
```
**結論：** RSS 是「一串 `<item>` 清單」——C2 段就是靠它拿到「有哪些文章 + 網址」。
</details>

In [ ]:
# 📝 小作業 A：把上限調大，多看一點，數數有幾個 <item>
# 你的答案：
print(r.text[:____])   # TODO：想看更多，把字數上限調大（本來 500）

---
## B・數據路：證交所 API 抓股價 → `prices.csv`

抓股價我們**不爬網頁、直接打 API**——證交所（TWSE）是政府公開資料，提供官方查詢 API，格式乾淨、本來就歡迎程式來拿。**有 API 就用 API、別硬爬**，這是爬蟲第一守則。

> 💬 `params` ＝ 把查詢條件（要哪檔、哪個月）掛到網址後面；`r.json()` ＝ 把回應那串 JSON 文字轉成你熟的 dict。

**預期輸出：** `stat： OK`、10 個欄位名、第一個交易日那列。⚠️ 行情數字是當日快照、**你跑的一定不同**。

In [ ]:
import requests

# 證交所「個股日成交資訊」API：date=查詢月份、stockNo=股票代號（2330 只是查詢參數）
r = requests.get("https://www.twse.com.tw/exchangeReport/STOCK_DAY",
                 ____={"response": "json", "date": "20260701", "stockNo": "2330"})   # TODO：把查詢條件掛上去的參數名
j = r.____()                      # TODO：把回應從 JSON 文字轉成 dict 的方法
print("stat：", j["stat"])        # "OK" = 查詢成功
print("欄位：", j["fields"])
print("第一列：", j["data"][0])   # 💡 j["data"] 是一列一列的資料，[0]＝第一個交易日

### B・變成表格

`DataFrame` ＝ pandas 幫你把資料組成「像 Excel 的表格」。

**預期輸出：** 一張表格（前幾列）。

In [ ]:
import pandas as pd

df_price = pd.____(j["data"], columns=j["fields"])   # TODO：pandas 把資料組成「表格」的東西（大小寫要對）
print(df_price.head())

### B・存成 `prices.csv`

`df` 可以匯出成很多格式（`.csv`、`.xlsx`…）；最通用的是 `.csv`。這個檔就是後面 pandas / KNN / 決策樹 的原料。

**預期輸出：**「已存 prices.csv」。

In [ ]:
print("共", len(df_price), "個交易日")
df_price.____("prices.csv", index=False)   # TODO：把表格存成 .csv 檔的方法
print("已存 prices.csv（數據路的原料）")

### B・取值小知識：`.iloc`

`.iloc[0]` ＝ 用「第幾列」的位置取值（`0` ＝第一列）。等一下用它抓出兩個「髒資料」給你看。

### B・當眾抓兩個「髒資料」（模組 2 的鉤子）

真實世界抓回來的資料**永遠是髒的**。指出兩個等一下 pandas 要修的地方。

**預期輸出：** 一個民國日期字串、一個含逗號的字串、型別是 `str`。

In [ ]:
print(df_price["日期"].____[0])       # TODO：用「第幾列」的位置取值（0＝第一列）
print(df_price["成交股數"].____[0])   # 💡 '37,544,470' ← 含千分位逗號，其實是字串不是數字，要修
print(type(df_price["成交股數"].____[0]))   # 💡 <class 'str'>——直接拿去算平均會爆錯
# 今天先不修！洗乾淨（民國轉西元、逗號轉數字）正是模組 2 pandas 要教的第一件事。

### 📝 小作業 B

1. 把 `stockNo` 換成別檔（例如 `"2317"` 鴻海），重跑抓成表格——**整段解析 code 一行都不用動**（這就是 API 的好處：條件是參數）。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
r2 = requests.get("https://www.twse.com.tw/exchangeReport/STOCK_DAY",
                  params={"response": "json", "date": "20260701", "stockNo": "2317"})
j2 = r2.json()
df2 = pd.DataFrame(j2["data"], columns=j2["fields"])
print(df2.head())
```
**結論：** 換股票只動 `stockNo` 參數，解析邏輯完全不變——這就是 API 比爬 HTML 穩的原因。
</details>

In [ ]:
# 📝 小作業 B：把 stockNo 換成 "2317"（鴻海），其他不用動
# 你的答案：
r2 = requests.get("https://www.twse.com.tw/exchangeReport/STOCK_DAY",
                  params=____)   # TODO：填 response/date/stockNo，stockNo 換成 "2317"
j2 = r2.json()
df2 = pd.DataFrame(j2["data"], columns=j2["fields"])
print(df2.head())
print("共", len(df2), "個交易日")

---
## C1・先找 API：用 F12 → Network 找出鉅亨新聞 API

要「財經新聞文字」之前，先做爬蟲工程師真正值錢的判斷：**這網站有沒有 API？** 而且要會**自己把 API 找出來**。跟著老師在自己的瀏覽器點一遍（這一段是滑鼠操作、不是 code）：

1. 開鉅亨網台股新聞列表頁 `https://news.cnyes.com/news/cat/tw_stock`
2. 按 **F12** → 點上排的 **Network（網路）** 分頁
3. 點 **Fetch/XHR** 篩選鈕（只看「程式去要資料」的請求）
4. **按 F5 重整頁面**——Network 會刷出一串請求
5. 找 **Name 有 `newslist`、Domain 是 `api.cnyes.com`** 的那條，點它 → 右邊 **Preview** 看到一坨 JSON，裡面每筆有 `title`、`content`——**這就是新聞的 API！**
6. 右鍵 → Copy → Copy link address，就是完整 API 網址

**讀 JSON 摸參數：** 回傳頂層有 `total`（總則數）、`last_page`（分幾頁）、`next_page_url`（下一頁）——所以「換頁就是改 `page` 參數、一頁幾則改 `limit`」。

**預期輸出：** 總則數、拿到幾則、第一則標題。

In [ ]:
import requests

# 剛剛用 F12 Network 找到的 API：newslist=新聞清單、tw_stock=台股分類、limit=一頁幾則
r = requests.get("https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
                 params=____)                 # TODO：查詢參數，一次拿 5 則（想想 B 段 params 長怎樣）
j = r.json()
data = j["items"]["data"]                     # 💡 新聞清單住在 items → data（F12 Preview 裡看到的路徑）
print("總則數 total：", j["items"]["total"])
print("這批拿到：", len(data), "則")
print("第一則標題：", data[0]["title"])

### C1・content 是「髒的」——先看它原本長怎樣

先把第一則的 `content` 印出來，親眼看它是「網頁原始碼格式」（`&lt;p&gt;` 其實是 `<p>`）。

**預期輸出：** 一段開頭有 `&lt;p&gt;` 的文字。

In [ ]:
import html, re

raw = data[0]["content"]
raw    # 💡 最後一行不寫 print，讓 Jupyter 直接把它攤開——看清楚裡面夾著 &lt;p&gt; 這種標籤

### C1・清理成純文字

兩步：① `html.unescape` 把 `&lt;` 這種「跳脫字元」還原成 `<`；② `re.sub` 把 `<p>` 之類標籤刮掉。

**預期輸出：** 乾淨的一段中文（沒有 `<p>`、沒有 `&lt;`）。

In [ ]:
# 兩步清理（這兩行照抄即可，正則不是今天重點）
text = html.unescape(raw)                      # 💡 &lt;p&gt; → <p>
text = re.sub(r"<[^>]+>", "", text)            # 💡 拿掉所有 <...> 標籤，只留純文字
text

### C1・組成 (date, title, text) → 存 `news_raw.csv`

一則一則跑過 `data`，把日期、標題、清理後的內文、字數組成一列。這個檔就是後面情緒打分要吃的貨。

**預期輸出：** 迴圈印出每則標題 → 一張表 →「已存 news_raw.csv」。

In [ ]:
import datetime, html, re
import pandas as pd

In [ ]:
rows = []
for it in ____:                                # TODO：要一則一則跑過哪個清單？（C1 開頭抓到的那批新聞）
    # publishAt 是「Unix 時間戳」（一個大數字），datetime 幫我們轉成看得懂的日期
    date = datetime.datetime.fromtimestamp(it["publishAt"]).strftime("%Y-%m-%d")
    clean = re.sub(r"<[^>]+>", "", html.unescape(it["content"]))
    rows.append({"date": date, "title": it["title"], "text": clean, "lengths": len(clean)})  # 💡 字數在這裡一起算好
    print(it["title"])

In [ ]:
df_news = pd.____(rows)    # TODO：pandas 把資料組成「表格」的東西（大小寫要對）
df_news

In [ ]:
df_news.____("news_raw.csv", index=False, encoding="utf-8-sig")   # TODO：把表格存成 .csv 檔的方法
print("已存 news_raw.csv（文字路正式貨源）")

### 📝 小作業 C1

1. 把 `limit` 改成 `10`，多抓幾則。
2. 印出第一則清理後的 `text` 前 100 字，**肉眼確認清理乾淨了**（沒殘留 `<p>` 或 `&lt;`）。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
r3 = requests.get("https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
                  params={"limit": 10})
data10 = r3.json()["items"]["data"]
print("這批拿到：", len(data10), "則")
print(re.sub(r"<[^>]+>", "", html.unescape(data10[0]["content"]))[:100])
```
**結論：** `total` 有好幾百則，你想抓幾則都行（改 `limit`）；清理後應該是乾淨中文、沒有標籤殘留。
</details>

In [ ]:
# 📝 小作業 C1：limit 改 10、再印第一則清理後前 100 字
# 你的答案：
r3 = requests.get("https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
                  params=____)   # TODO：這次一次拿 10 則
data10 = r3.json()["items"]["data"]
print("這批拿到：", len(data10), "則")

clean0 = re.sub(r"<[^>]+>", "", html.unescape(data10[0]["content"]))
clean0[:100]

---
## C2・沒 API 的網站怎麼辦：中央社 RSS + BeautifulSoup 兩層

不是每個網站都給 API。**中央社沒有公開的內文 API**，這時才用「兩層結構」爬 HTML：**RSS 列表拿「有哪些文章 + 網址」→ 一篇一篇進內頁拿「內文」**。這段是真正在爬 HTML、也是今天的爬蟲真功夫。

In [ ]:
import requests
from bs4 import BeautifulSoup

res = requests.get("https://feeds.feedburner.com/rsscna/finance")

### C2・小對照：不是每個回應都能 `.json()`

C1 的鉅亨是 API、回 JSON，所以 `.json()` 好用。這裡的 RSS 回的是 **XML**，硬用 `.json()` 會**報錯**——親眼看一次。

In [ ]:
# 故意用 .json() 看它報錯（RSS 是 XML、不是 JSON）
try:
    res.json()
except Exception as e:
    print("❌ 用 .json() 解析失敗：", e)
    print("→ 這回傳的是 XML（RSS），不是 JSON，所以不能 .json()；要用 res.text + BeautifulSoup")

### C2・那就看 `res.text`（一坨 HTML/XML 文字）

**預期輸出：** `<?xml ...>` 開頭、一串 `<item>` 的文字。

In [ ]:
res.text[:800]    # 💡 前 800 字：一串 <item>，每個裡面有 <title> 和 <link>

### C2・交給 BeautifulSoup 解析

`BeautifulSoup` 把「一坨文字」變成「可以查找的物件」。

> 💬 HTML/XML 由**標籤（tag）** 組成，例 `<title>…</title>`；標籤上可掛 **class**（分類名牌）和 **attr**（屬性）。老師會簡單帶一下。

**預期輸出：** 一份排整齊的解析結果（前一段）。

In [ ]:
rss = BeautifulSoup(res.text, "html.parser")   # 💡 解析器用內建 html.parser，不用另外裝 lxml
print(rss.prettify()[:600])                    # 💡 排整齊看前 600 字，感受一下 <item> 的結構

### C2・抽出前 3 則（標題, 內頁網址）

每則文章是一個 `<item>`，裡面 `<title>` 是標題、`<link>` 是內頁網址。只取前 3 則（禮貌頻率）。

**預期輸出：** 3 行「標題 → 網址」。

In [ ]:
items = []
for it in rss.find_all(____)[:3]:                # TODO：RSS 裡「每則文章」的標籤名
    title = it.find(____).get_text(strip=True)   # TODO：放「標題」的標籤名
    # ⚠️ 這個網址標籤有自閉合小地雷，值會跑到 next_sibling——整行照抄即可
    link = (it.find(____).next_sibling or "").strip() or it.find(____).get_text(strip=True)  # TODO：放「內頁網址」的標籤（兩處填一樣）
    items.append((title, link))

for t, u in items:
    print(t[:30], "→", u)

### C2・先看一下要抓的內頁網址

**預期輸出：** 第一則的內頁網址字串。

In [ ]:
url = items[0][1]
url    # 💡 等一下就進這個網址抓內文

### C2・這次「不表明身分」抓內頁，看會怎樣

還記得 A 段抓 RSS 不用身分也行嗎？**內頁不一樣**——先不帶身分試試。

**預期輸出：** 一段很短、看不到新聞的內容（被擋了）。

In [ ]:
# 先不帶身分抓內頁
page_html = requests.get(url).text
print("沒帶身分抓到的長度：", len(page_html))   # 💡 跟下一格 9 萬多字對比——這裡短很多
page_html[:200]                                # 💡 而且根本不是新聞內文——被擋住了（狀態其實是 403）

### C2・帶上「身分」（User-Agent）就抓得到了

`User-Agent` ＝ 請求裡「我是誰」的自我介紹欄位。帶上瀏覽器字樣、有禮貌地表明「我跟瀏覽器同款」，內頁就給你了。

**預期輸出：** 這次是完整的一大段 HTML。

In [ ]:
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}   # 表明身分的瀏覽器標頭
page_html = requests.get(url, headers=UA).text
print("這次抓到了，長度：", len(page_html))
print(page_html[:300])    # 💡 對照上一格：現在是完整 HTML 了。爬內頁記得帶 headers=UA

### C2・看 soup、找內文住在哪個標籤

把內頁交給 BeautifulSoup，看它的結構，才知道 `SELECTORS`（標題/內文/時間）各要抓哪個標籤。

**預期輸出：** 一段排整齊的 HTML（前一段）。

In [ ]:
soup = BeautifulSoup(page_html, "html.parser")
print(soup.prettify()[:600])    # 💡 排整齊看前 600 字；實務上用瀏覽器 F12 找內文住哪個標籤更快

### C2・用 SELECTORS 撈標題 / 時間 / 內文

**預期輸出：** 標題、時間、內文段落數、第一段前 50 字。⚠️ 段數/內容依每篇而定。

In [ ]:
SELECTORS = {"標題": "h1", "內文": "div.paragraph p", "時間": "div.updatetime"}   # 💡 改版只改這一處

print("標題：", soup.select_one(SELECTORS["標題"]).get_text(strip=True))
print("時間：", soup.select_one(SELECTORS["時間"]).get_text(strip=True))
paras = soup.select(SELECTORS["內文"])           # 💡 select 複數＝撈「所有」內文段落，回來是清單
print("內文段落數：", len(paras))
print("第一段前 50 字：", paras[0].get_text(strip=True)[:50])

### C2・迴圈 3 篇 → 印出驗證（每頁睡 1 秒）

**禮貌頻率**：每抓一頁 `sleep(1)`。算給你看：30 人 × 3 頁 = 90 個請求打到人家網站——不睡就像攻擊。

**預期輸出：** 三欄表 + 各篇字數。⚠️ 爬出來的形狀跟 C1 鉅亨 API 拿的**一模一樣**（date/title/text）——兩路殊途同歸。

In [ ]:
import time
import pandas as pd

rows = []
for title, url in items:
    page_html = requests.get(url, headers=UA, timeout=15).text
    soup = BeautifulSoup(page_html, "html.parser")
    rows.append({
        "date":  soup.select_one(SELECTORS["時間"]).get_text(strip=True),
        "title": soup.select_one(SELECTORS["標題"]).get_text(strip=True),
        "text":  "".join(p.get_text(strip=True) for p in soup.select(SELECTORS["內文"])),
    })
    time.sleep(1)                                # 💡 禮貌頻率：每抓一頁睡 1 秒（必寫）——你是客人，別當壞鄰居

df_cna = pd.DataFrame(rows)                       # 💡 叫 df_cna，別蓋掉 C1 的 df_news（那是正式貨源）
print(df_cna[["date", "title"]])

lengths = []
for t in df_cna["text"]:
    lengths.append(len(t))
print("各篇內文字數：", lengths)
print("✅ 兩層結構爬通了——這就是沒 API 時的真功夫（正式 news_raw.csv 已在 C1 產好）")

### 📝 小作業 C2

1. 印出第一則的內文前 100 字，**肉眼確認抓到的是新聞正文**、不是廣告或選單。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
print(df_cna["text"].iloc[0][:100])
# 若是新聞正文＝成功；若是「登入/訂閱/選單」＝selector 抓錯，回 F12 看內文住哪個標籤、只改 SELECTORS 一處。
```
</details>

In [ ]:
# 📝 小作業 C2：印第一則內文前 100 字，肉眼抽查是不是新聞正文
# 你的答案：
print(df_cna["text"].iloc[0][:____])   # TODO：印前 100 字

---
## 🔑 收尾：今天的交付物 + 一個帶得走的判斷力

- ✅ `prices.csv`（B 段・證交所 API・數據路原料）
- ✅ `news_raw.csv`（C1・鉅亨 API・文字路正式貨源）

🧭 **比檔案更值錢的：** 你今天走了一次爬蟲工程師的完整判斷鏈——**要資料 → 先問有沒有 API → 用 F12 找 API（C1 鉅亨）→ 沒 API 才爬 HTML（C2 中央社）**。API 路省又穩、爬蟲路是沒 API 時的真功夫，兩種你都會了。